# Обучение модели зацепок для Skalyk

Этот ноутбук скачает набор с разметкой контуров, дообучит RF-DETR Seg Nano на бесплатной GPU Kaggle и экспортирует модель для запуска прямо в браузере. Фото пользователей при распознавании не отправляются в сеть.

Набор: [Climbing Hold Detection v26](https://universe.roboflow.com/climbtag/climbing-hold-detection-7uehq/dataset/26), лицензия CC BY 4.0. В нём отмечены зацепки и объёмные элементы, но нет стартов и финишей.

Перед запуском в Kaggle включите Internet и выберите бесплатный GPU в Settings. Доступность GPU зависит от квоты аккаунта.

In [ ]:
%pip -q install "rfdetr[train,loggers,onnx]" roboflow

import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('В настройках Kaggle включите GPU и запустите ячейку заново.')

## 1. Скачать размеченные изображения

Создайте бесплатный API key в Roboflow и вставьте его в появившееся скрытое поле. Ячейка не печатает и не сохраняет ключ.

In [ ]:
from getpass import getpass
from pathlib import Path
from roboflow import Roboflow

api_key = getpass('Roboflow API key (ввод скрыт): ')
project = Roboflow(api_key=api_key).workspace('climbtag').project('climbing-hold-detection-7uehq')
dataset = project.version(26).download('coco', location='/kaggle/working/climbing-holds-v26')
dataset_dir = Path(dataset.location)
print('Dataset folder:', dataset_dir)
print('COCO train annotations:', (dataset_dir / 'train' / '_annotations.coco.json').exists())
del api_key

## 2. Дообучить модель

Начинаем с предварительно обученных весов, а модель учится сегментировать два класса: зацепки и объёмные элементы. При нехватке GPU-памяти уменьшите `batch_size` до 2.

In [ ]:
from rfdetr import RFDETRSegNano

output_dir = Path('/kaggle/working/rfdetr-climbing')
model = RFDETRSegNano()
model.train(
    dataset_dir=str(dataset_dir),
    output_dir=str(output_dir),
    epochs=30,
    batch_size=4,
    resolution=312,
    lr=1e-4,
    early_stopping=True,
    early_stopping_patience=8,
    run_test=True,
)

checkpoint = output_dir / 'checkpoint_best_total.pth'
if not checkpoint.exists():
    raise FileNotFoundError(f'Не найден лучший checkpoint: {checkpoint}')
print('Best checkpoint:', checkpoint)

## 3. Экспортировать компактную модель

INT8-экспорт уменьшает размер модели примерно втрое, но может немного снизить точность. Для калибровки берём изображения из validation split.

In [ ]:
model = RFDETRSegNano(pretrain_weights=str(checkpoint))
validation_images = dataset_dir / 'valid'
if not validation_images.exists():
    validation_images = dataset_dir / 'valid' / 'images'

model.export(
    format='onnx',
    quantization='int8',
    calibration_data=str(validation_images),
)

onnx_candidates = list(Path('output').rglob('*int8*.onnx'))
if not onnx_candidates:
    onnx_candidates = list(output_dir.rglob('*int8*.onnx'))
if not onnx_candidates:
    raise FileNotFoundError('Не найден INT8 ONNX-файл после экспорта.')

model_file = onnx_candidates[0]
delivery_file = Path('/kaggle/working/climbing-holds-int8.onnx')
delivery_file.write_bytes(model_file.read_bytes())
print('Ready for the app:', delivery_file)
print('Size, MiB:', round(delivery_file.stat().st_size / (1024 * 1024), 1))

## 4. Скачать файл модели

Скачайте `climbing-holds-int8.onnx` из панели Output в Kaggle. В проекте Skalyk положите файл в `public/models/climbing-holds-int8.onnx`, затем соберите и опубликуйте приложение. Сохраните ссылку на источник данных и лицензию при публикации: CC BY 4.0, автор climbtag.

Приложение предлагает старт снизу и финиш сверху, но просит проверить их вручную. Набор не содержит меток start/finish, поэтому модель не обучается предсказывать их напрямую.